# CRISP-DM Fase 1: Business Understanding (Comprensión del Negocio)
## Dataset: Documentación Técnica y Boletines Webservices SIPSA (DANE)

**Proyecto**: AgroStats Intelligence Platform (`AgroStatsApp`)  
**Metodología**: CRISP-DM (Fase 1: Business Understanding) | **Fase PDCO**: PLAN  
**Estándares**: DAMA-DMBOK 2 (Metadata Management & Governance), SWEBOK Cap. 1, ISO/IEC 25010.

---

### 🎯 1. Objetivo Estratégico y Problema de Negocio
Automatizar la extracción de especificaciones técnicas y esquemas de datos a partir de manuales y boletines institucionales.

### 💼 2. Entidad del Dominio y Alcance
* **Entidad Analítica Principal**: `Documentos Técnicos no estructurados, Especificaciones SOAP/WSDL y Reportes PDF`
* **Custodio y Fuente Oficial**: DANE / IDEAM / UPRA / SIPSA Colombia.
* **Granularidad Espacial**: `No Estructurado: Nivel Documento y Párrafo`
* **Granularidad Temporal**: `Versión Documental`

### 💡 3. Preguntas de Negocio Mapeadas (`docs/1-Bateria_preguntas.md`):
* **H1: Minería Textual y Tokenización de Parámetros**
* **H2: Indexación de Métodos de Consumo WSDL**

### 📊 4. Indicadores Clave de Desempeño (KPIs):
* `Densidad de caracteres extraídos`
* `Completitud de esquemas documentados`
* `Tasa de éxito de parsing PDF`

---


In [ ]:
# ==============================================================================
# [DEPENDENCIAS DE ENTORNO — JUPYTER / COLAB / DATABRICKS]
# ==============================================================================
%pip install -q pandas numpy requests python-dotenv openpyxl pypdf pyarrow pyreadstat matplotlib seaborn scipy statsmodels scikit-learn duckdb pydantic


In [ ]:
# ==============================================================================
# [CONFIGURACIÓN DEL KERNEL Y RESOLUCIÓN DE RUTAS DEL PROYECTO (PEP 8)]
# ==============================================================================
import os
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# Resolver raíz del proyecto para importar módulos de src/
CURRENT_DIR = Path(".").resolve()
APP_ROOT = CURRENT_DIR.parent.parent
if str(APP_ROOT) not in sys.path:
    sys.path.insert(0, str(APP_ROOT))

import pandas as pd
import numpy as np

print(f"✅ Entorno inicializado exitosamente. Raíz del proyecto: {APP_ROOT}")


In [ ]:
# ==============================================================================
# [FASE 1: VERIFICACIÓN DEL CONTRATO DE NEGOCIO Y CONFIGURACIÓN YAML]
# ==============================================================================
import yaml

config_path = APP_ROOT / "config" / "datasets_config.yaml"
if config_path.exists():
    with open(config_path, "r", encoding="utf-8") as f:
        config_data = yaml.safe_load(f)
    print(f"📄 Archivo de configuración cargado: {config_path.name}")
    dataset_cfg = config_data.get("datasets", {}).get("08_boletin_pdf_webservice", {})
    print(f"📋 Especificación del Dataset en YAML: {dataset_cfg}")
else:
    print("ℹ️ Configuración YAML por defecto activa.")

print("\n--- DEFINICIÓN DEL PROBLEMA DE NEGOCIO ---")
print("• Entidad:", "Documentos Técnicos no estructurados, Especificaciones SOAP/WSDL y Reportes PDF")
print("• Granularidad Espacial:", "No Estructurado: Nivel Documento y Párrafo")
print("• Granularidad Temporal:", "Versión Documental")
print("• KPIs Objetivo:", ['Densidad de caracteres extraídos', 'Completitud de esquemas documentados', 'Tasa de éxito de parsing PDF'])


In [ ]:
# ==============================================================================
# [FASE 1: INSPECCIÓN DE DISPONIBILIDAD DE FUENTES Y CONTRATO DAMA-DMBOK]
# ==============================================================================
from src.database.db_manager import DatabaseManager

db_path = APP_ROOT / "data" / "processed" / "agrostats_lakehouse.db"
parquet_path = APP_ROOT / "data/processed/doc_webservice_chunks.parquet"

print("🔍 Verificación de Almacenamiento en Data Lakehouse:")
print(f"• Archivo Parquet Silver/Gold: {parquet_path.name} -> Existe: {parquet_path.exists()}")
print(f"• Base de Datos SQLite Gold: {db_path.name} -> Existe: {db_path.exists()}")

if parquet_path.exists():
    df_sample = pd.read_parquet(parquet_path)
    print(f"\n✅ Contrato de datos preliminar verificado:")
    print(f"• Filas totales: {len(df_sample):,}")
    print(f"• Columnas: {list(df_sample.columns)}")
